In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG = "project_lakehouse_telemetria_ele"
SRC  = f"{CATALOG}.bronze.telemetria_raw"
TGT  = f"{CATALOG}.silver.telemetria_clean"
QUAR = f"{CATALOG}.silver.telemetria_quarentena"

cols = ["device_id", "event_ts", "tensao_v", "corrente_a", "temperatura_c", "alarme"]
raw = spark.table(SRC)

In [0]:
# 1) Hash da linha completa (nulos entram como marcador)
row_hash = F.sha2(
    F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("<NULL>")) for c in cols]), 256)

In [0]:
# 2) Deduplicação sobre o dado bruto
dedup = raw.withColumn("row_hash", row_hash).dropDuplicates(["row_hash"])

In [0]:
# 3) Tipagem segura (inválido vira NULL, sem quebrar em modo ANSI)
typed = (dedup
    .withColumn("event_ts", F.expr("try_cast(event_ts as timestamp)"))
    .withColumn("tensao_v", F.expr("try_cast(tensao_v as double)"))
    .withColumn("corrente_a", F.expr("try_cast(corrente_a as double)"))
    .withColumn("temperatura_c", F.expr("try_cast(temperatura_c as double)")))

In [0]:
# 4) Flags e tratamento da sentinela
silver = (typed
    .withColumn("corrente_sentinela", F.coalesce(F.col("corrente_a") == -999, F.lit(False)))
    .withColumn("corrente_a",
        F.when(F.col("corrente_sentinela"), F.lit(None).cast("double")).otherwise(F.col("corrente_a")))
    .withColumn("tensao_ausente", F.col("tensao_v").isNull())
    .withColumn("temp_pico", F.coalesce(F.col("temperatura_c") > 100, F.lit(False)))
    .withColumn("data_evento", F.to_date("event_ts"))
    .select("row_hash", *cols, "corrente_sentinela", "tensao_ausente",
            "temp_pico", "data_evento", "_ingest_ts", "_source_file"))

In [0]:
# 5) Quarentena idempotente
valid   = silver.filter(F.col("device_id").isNotNull() & F.col("event_ts").isNotNull())
invalid = silver.filter(F.col("device_id").isNull() | F.col("event_ts").isNull())

if spark.catalog.tableExists(QUAR):
    (DeltaTable.forName(spark, QUAR).alias("t")
        .merge(invalid.alias("s"), "t.row_hash = s.row_hash")
        .whenNotMatchedInsertAll().execute())
else:
    invalid.write.format("delta").saveAsTable(QUAR)


In [0]:
# 6) Escrita idempotente da Silver
if spark.catalog.tableExists(TGT):
    (DeltaTable.forName(spark, TGT).alias("t")
        .merge(valid.alias("s"), "t.row_hash = s.row_hash")
        .whenNotMatchedInsertAll().execute())
else:
    valid.write.format("delta").saveAsTable(TGT)

In [0]:
print("Silver:", spark.table(TGT).count())